# Ingestão dos dados e criação da camada bronze:

In [30]:
from pathlib import Path
from urllib.parse import urlparse
import requests

In [31]:
BRONZE_DIR = Path.cwd().parent / "data" / "bronze"
HEADERS = {"User-Agent": "DataIngestionPipeline/1.0"}

# Intervalo de analise dos dados
START_YEAR = 2015
END_YEAR = 2025

Puxando dados por API REST:

In [32]:
# Série histórica por intervalo de anos
latitude = -15.78
longitude = -47.92

url_openmeteo = "https://archive-api.open-meteo.com/v1/archive"
params_clima = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": f"{START_YEAR}-01-01",
    "end_date": f"{END_YEAR}-12-31",
    "daily": "temperature_2m_mean,precipitation_sum",
    "timezone": "America/Sao_Paulo",
}

print(f"Buscando clima na API para o período {START_YEAR}-{END_YEAR}...")
resp_clima = requests.get(url_openmeteo, params=params_clima, headers=HEADERS, timeout=30)
resp_clima.raise_for_status()

dir_clima_api = BRONZE_DIR / "openmeteo_clima"
dir_clima_api.mkdir(parents=True, exist_ok=True)

path_clima = dir_clima_api / f"clima_historico_{START_YEAR}_{END_YEAR}.json"
path_clima.write_bytes(resp_clima.content)

print(f"Salvo em: {path_clima}")

Buscando clima na API para o período 2015-2025...
Salvo em: /home/gogdar/code/projects/el-nino-energy-effects-brazil-analysis/data/bronze/openmeteo_clima/clima_historico_2015_2025.json


In [33]:
# Consulta ao catálogo CKAN
url_aneel_api = "https://dadosabertos.aneel.gov.br/api/3/action/package_show"

print("Consultando API da ANEEL...")
resp_aneel = requests.get(url_aneel_api, params={"id": "samp"}, headers=HEADERS, timeout=30)
resp_aneel.raise_for_status()

resources = resp_aneel.json().get("result", {}).get("resources", [])
dir_aneel = BRONZE_DIR / "aneel_samp"
dir_aneel.mkdir(parents=True, exist_ok=True)

for res in resources:
    if res.get("format", "").upper() == "CSV":
        file_url = res["url"]
        filename = Path(urlparse(file_url).path).name or "samp.csv"
        
        print(f"Baixando base de energia: {filename}...")
        file_resp = requests.get(file_url, headers=HEADERS, timeout=120)
        file_resp.raise_for_status()
        
        out_path = dir_aneel / filename
        out_path.write_bytes(file_resp.content)

print("Downloads da ANEEL concluídos.")

Consultando API da ANEEL...
Baixando base de energia: samp-2003.csv...
Baixando base de energia: samp-2004.csv...
Baixando base de energia: samp-2005.csv...
Baixando base de energia: samp-2006.csv...
Baixando base de energia: samp-2007.csv...
Baixando base de energia: samp-2008.csv...
Baixando base de energia: samp-2009.csv...
Baixando base de energia: samp-2010.csv...
Baixando base de energia: samp-2011.csv...
Baixando base de energia: samp-2012.csv...
Baixando base de energia: samp-2013.csv...
Baixando base de energia: samp-2014.csv...
Baixando base de energia: samp-2015.csv...
Baixando base de energia: samp-2016.csv...
Baixando base de energia: samp-2017.csv...
Baixando base de energia: samp-2018.csv...
Baixando base de energia: samp-2019.csv...
Baixando base de energia: samp-2020.csv...
Baixando base de energia: samp-2021.csv...
Baixando base de energia: samp-2022.csv...
Baixando base de energia: samp-2023.csv...
Baixando base de energia: samp-2024.csv...
Baixando base de energia: 

Puxando dados por download direto:

In [34]:
# NOAA Bulk
url_noaa = "https://psl.noaa.gov/data/correlation/oni.data"

print("Baixando série completa do El Niño (NOAA)...")
resp_noaa = requests.get(url_noaa, headers=HEADERS, timeout=30)
resp_noaa.raise_for_status()

dir_noaa = BRONZE_DIR / "noaa_oni"
dir_noaa.mkdir(parents=True, exist_ok=True)

path_noaa = dir_noaa / "oni_1950_presente.txt"
path_noaa.write_bytes(resp_noaa.content)

print(f"Salvo em: {path_noaa}")

Baixando série completa do El Niño (NOAA)...
Salvo em: /home/gogdar/code/projects/el-nino-energy-effects-brazil-analysis/data/bronze/noaa_oni/oni_1950_presente.txt


In [35]:
# EPE Bulk
url_epe = (
    "https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/"
    "dados-abertos/Documents/Dados_abertos_Consumo_Mensal.xlsx"
)

print("Baixando planilha histórica da EPE...")
resp_epe = requests.get(url_epe, headers=HEADERS, timeout=60)
resp_epe.raise_for_status()

dir_epe = BRONZE_DIR / "epe_consumo"
dir_epe.mkdir(parents=True, exist_ok=True)

path_epe = dir_epe / "consumo_mensal_historico.xlsx"
path_epe.write_bytes(resp_epe.content)

print(f"Salvo em: {path_epe}")

Baixando planilha histórica da EPE...
Salvo em: /home/gogdar/code/projects/el-nino-energy-effects-brazil-analysis/data/bronze/epe_consumo/consumo_mensal_historico.xlsx
